In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef, confusion_matrix, roc_auc_score, cohen_kappa_score
from collections import OrderedDict
rng = np.random.default_rng(42)

In [ ]:
# 1) Load
df = pd.read_csv("generated_label.csv")
df = df.dropna(subset=["label"])
y = df["label"].astype(int).values

models = OrderedDict({
    "gemini": pd.to_numeric(df["gemini"], errors="coerce").fillna(-1).astype(int).values,
    "chatgpt": pd.to_numeric(df["Chatgpt"], errors="coerce").fillna(-1).astype(int).values,
    "deepseek": pd.to_numeric(df["Deepseek"], errors="coerce").fillna(-1).astype(int).values,
})


In [ ]:
mask = (models["gemini"] != -1) & (models["chatgpt"] != -1) & (models["deepseek"] != -1)
y = y[mask]
for k in models:
    models[k] = models[k][mask]


In [ ]:
# 2) Metrics per model
def metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    # AUROC only if both classes present in preds
    try:
        auroc = roc_auc_score(y_true, y_pred)
    except ValueError:
        auroc = np.nan
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    return {"acc": acc, "f1_w": f1_w, "f1_m": f1_m, "mcc": mcc, "auroc": auroc, "cm": cm}

summary = {}
for name, yp in models.items():
    summary[name] = metrics(y, yp)

In [ ]:
# 3) Bootstrap 95% CI for accuracy & F1(macro)
def bootstrap_ci(y_true, y_pred, fn, B=2000):
    n = len(y_true); stats = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        stats.append(fn(y_true[idx], y_pred[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return lo, hi

for name, yp in models.items():
    acc_lo, acc_hi = bootstrap_ci(y, yp, accuracy_score)
    f1m_lo, f1m_hi = bootstrap_ci(y, yp, lambda a,b: precision_recall_fscore_support(a,b,average="macro",zero_division=0)[2])
    summary[name]["acc_ci"] = (acc_lo, acc_hi)
    summary[name]["f1m_ci"] = (f1m_lo, f1m_hi)

In [ ]:
# 4) Agreement (κ) between models
agreements = {}
names = list(models.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a = cohen_kappa_score(models[names[i]], models[names[j]])
        agreements[f"{names[i]} vs {names[j]}"] = a

In [ ]:
# 5) Optional: McNemar’s test (ChatGPT vs DeepSeek example)
from statsmodels.stats.contingency_tables import mcnemar
def mcnemar_p(y_true, y1, y2):
    # contingency of disagreements
    a = np.sum((y1 == y_true) & (y2 != y_true))
    b = np.sum((y2 == y_true) & (y1 != y_true))
    table = [[0, a],
             [b, 0]]
    return mcnemar(table, exact=False, correction=True).pvalue

p_chatgpt_vs_deepseek = mcnemar_p(y, models["chatgpt"], models["deepseek"])

In [ ]:
# 6) Print nicely
def fmt(ci): return f"[{ci[0]:.3f}, {ci[1]:.3f}]"
for name, s in summary.items():
    print(f"\n== {name.upper()} ==")
    print(f"Acc: {s['acc']:.3f}  95%CI {fmt(s['acc_ci'])}")
    print(f"F1(macro): {s['f1_m']:.3f}  95%CI {fmt(s['f1m_ci'])}")
    print(f"F1(weighted): {s['f1_w']:.3f}   MCC: {s['mcc']:.3f}   AUROC: {s['auroc'] if s['auroc']==s['auroc'] else 'NA'}")
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:\n", s['cm'])
print("\nCohen's κ (agreement):")
for k,v in agreements.items():
    print(f"{k}: {v:.3f}")
print(f"\nMcNemar p-value (chatgpt vs deepseek): {p_chatgpt_vs_deepseek:.4f}")


== GEMINI ==
Acc: 0.631  95%CI [0.564, 0.698]
F1(macro): 0.615  95%CI [0.545, 0.686]
F1(weighted): 0.612   MCC: 0.315   AUROC: 0.6403475868967241
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[38 55]
 [11 75]]

== CHATGPT ==
Acc: 0.626  95%CI [0.553, 0.698]
F1(macro): 0.599  95%CI [0.524, 0.672]
F1(weighted): 0.595   MCC: 0.328   AUROC: 0.636721680420105
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[33 60]
 [ 7 79]]

== DEEPSEEK ==
Acc: 0.642  95%CI [0.570, 0.715]
F1(macro): 0.642  95%CI [0.569, 0.709]
F1(weighted): 0.642   MCC: 0.289   AUROC: 0.6440985246311578
Confusion matrix [ [TN, FP], [FN, TP] ]:
 [[56 37]
 [27 59]]

Cohen's κ (agreement):
gemini vs chatgpt: 0.627
gemini vs deepseek: 0.538
chatgpt vs deepseek: 0.406

McNemar p-value (chatgpt vs deepseek): 0.7794
